# Adım 1 — Veri setlerini tanıyalım (henüz model eğitmiyoruz)

**Amaç:** Hugging Face'teki prompt-injection veri setlerine bakmak: kaç satır var, sütunlar ne, etiket (0/1) ne demek, örnek cümleler neye benziyor?

**Nasıl çalıştırılır:** Üst menüden `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçin. Sonra hücreleri **sırayla**, her birinin solundaki ▶ düğmesine basarak çalıştırın.
Bir hücre hata verirse **durmayın**, hatayı kopyalayıp bana yazın. Kod, bir veri seti açılmazsa diğerlerine devam edecek şekilde yazıldı.

Sonunda **7. hücrenin çıktısını** (tabloyu ve örnekleri) kopyalayıp bana yapıştırın. Hangi 3 veri setini kullanacağımıza birlikte karar vereceğiz.

## 1) Ekran kartı var mı?

In [ ]:
import torch
# True yazıyorsa Colab ücretsiz ekran kartını (GPU) bize vermiş demektir
print("GPU var mı:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Kart:", torch.cuda.get_device_name(0))

## 2) Kütüphaneleri kuralım
`datasets` = Hugging Face'ten veri indirir. `pandas` = tablo olarak gösterir.

In [ ]:
!pip -q install datasets pandas

## 3) Hangi veri setlerine bakacağız?
Bunlar hocanın verdiği Hugging Face listesinden ve bilinen veri setlerinden seçilmiş **adaylar**. Hepsini kullanmayacağız, bakıp en uygun 3 tanesini seçeceğiz.
Format her zaman `sahip/ad` şeklindedir.

In [ ]:
adaylar = [
    "deepset/prompt-injections",
    "xTRam1/safe-guard-prompt-injection",
    "jackhhao/jailbreak-classification",
    "neuralchemy/Prompt-injection-dataset",
    "wambosec/prompt-injections",
    "guychuk/benign-malicious-prompt-classification",
]

## 4) Hepsini indirip özetini çıkaran yardımcı fonksiyon
Satır satır ne yaptığı yorumlarda yazıyor. **Okuyun**, sadece çalıştırmayın.

In [ ]:
from datasets import load_dataset
import pandas as pd

ozetler = []      # her veri setinin kısa özeti buraya girecek
yuklenenler = {}  # indirilen veri setlerini sonra kullanmak için saklıyoruz

for ad in adaylar:
    try:
        ds = load_dataset(ad)            # veri setini Hugging Face'ten indir
        yuklenenler[ad] = ds
        for bolum in ds.keys():          # 'train', 'test' gibi bölümler
            tablo = ds[bolum].to_pandas()
            ozetler.append({
                "veri_seti": ad,
                "bolum": bolum,
                "satir": len(tablo),
                "sutunlar": list(tablo.columns),
            })
    except Exception as hata:
        # İndirilemezse (izin gerekiyor, ad değişmiş vb.) durmayız, sadece not alırız
        ozetler.append({"veri_seti": ad, "bolum": "-", "satir": 0,
                        "sutunlar": "HATA: " + str(hata)[:80]})

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
ozet_tablosu = pd.DataFrame(ozetler)
ozet_tablosu

## 5) Etiket sütunu hangisi, kaç tane zararlı / zararsız?
Her veri setinde etiket sütununun adı farklı olabilir (`label`, `type`, ...). Aşağıdaki kod, tahmin ederek bulur ve sayar.

In [ ]:
etiket_adaylari = ["label", "labels", "type", "category", "class", "is_injection", "injection"]

for ad, ds in yuklenenler.items():
    print("=" * 70)
    print(ad)
    for bolum in ds.keys():
        tablo = ds[bolum].to_pandas()
        etiket_sutunu = next((s for s in tablo.columns if s.lower() in etiket_adaylari), None)
        print(f"  [{bolum}] sütunlar: {list(tablo.columns)}")
        if etiket_sutunu:
            print(f"  [{bolum}] '{etiket_sutunu}' sütunundaki değerler ve sayıları:")
            print(tablo[etiket_sutunu].value_counts().to_string())
        else:
            print(f"  [{bolum}] etiket sütunu otomatik bulunamadı, elle bakmak gerek")

## 6) Örnek cümlelere bakalım
Veri setinin neye benzediğini **gözle** görmek çok önemli. Her veri setinden 4 rastgele satır gösterir (uzun metinler 160 karakterde kesilir).

In [ ]:
for ad, ds in yuklenenler.items():
    bolum = list(ds.keys())[0]           # ilk bölüm (genelde 'train')
    tablo = ds[bolum].to_pandas()
    print("=" * 70)
    print(ad, f"({bolum})")
    ornek = tablo.sample(n=min(4, len(tablo)), random_state=1)
    for _, satir in ornek.iterrows():
        parcalar = [f"{k}: {str(v)[:160]!r}" for k, v in satir.items()]
        print("  - " + " | ".join(parcalar))

## 7) Her şeyi tek yerde toplayalım ✅
**Bu hücrenin çıktısını kopyalayıp bana yapıştırın.**

In [ ]:
print("ÖZET TABLO")
print(ozet_tablosu.to_string())
print()
print("Yüklenen veri setleri:", list(yuklenenler.keys()))
print("Yüklenemeyenler:", [a for a in adaylar if a not in yuklenenler])